In [1]:
import numpy as np
import pandas as pd
import matplotlib as plt
import seaborn as sns
import os

In [46]:
transaction = pd.read_csv("../data/raw/transaction_data.csv")
transaction['household_key'].nunique()

2500

In [31]:
transaction.head(100)

,household_key,BASKET_ID,DAY,PRODUCT_ID,QUANTITY,SALES_VALUE,STORE_ID,RETAIL_DISC,TRANS_TIME,WEEK_NO,COUPON_DISC,COUPON_MATCH_DISC
0,2375,26984851472,1,1004906,1,1.39,364,-0.60,1631,1,0.0,0.0
1,2375,26984851472,1,1033142,1,0.82,364,0.00,1631,1,0.0,0.0
2,2375,26984851472,1,1036325,1,0.99,364,-0.30,1631,1,0.0,0.0
3,2375,26984851472,1,1082185,1,1.21,364,0.00,1631,1,0.0,0.0
4,2375,26984851472,1,8160430,1,1.50,364,-0.39,1631,1,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
95,1060,26985040735,1,9553288,1,8.49,315,0.00,1251,1,0.0,0.0
96,1351,26985052379,1,903230,1,0.99,447,-0.30,1955,1,0.0,0.0
97,744,26985165432,1,5978648,0,0.00,31582,0.00,1119,1,0.0,0.0
98,212,26985205886,1,822346,1,1.25,288,-0.34,1341,1,0.0,0.0


In [3]:
frequency = transaction.groupby("household_key")["BASKET_ID"].nunique()

In [ ]:
frequency.head()

household_key
1      86
2      45
3      47
4      30
5      40
6     250
7      59
8     113
9      20
10      9
Name: BASKET_ID, dtype: int64

In [6]:
monetary = transaction.groupby("household_key")["SALES_VALUE"].sum()
monetary.head()

household_key
1    4330.16
2    1954.34
3    2653.21
4    1200.11
5     779.06
Name: SALES_VALUE, dtype: float64

In [10]:
last_day = transaction["DAY"].max()
print(last_day)
last_purchase = transaction.groupby("household_key")["DAY"].max()
recency = last_day - last_purchase
recency.head()

711


household_key
1     5
2    43
3     8
4    84
5     8
Name: DAY, dtype: int64

In [13]:
customer = pd.DataFrame()

customer["recency"] = recency
customer["frequency"] = frequency
customer["monetary"] = monetary
customer.head()
customer.shape

(2500, 3)

In [14]:
customer["avg_basket_value"] = customer["monetary"] / customer["frequency"]

In [ ]:
quantity = transaction.groupby("household_key")["QUANTITY"].sum()
customer["total_quantity"] = quantity

In [38]:
print(customer.head())

               recency  frequency  monetary  avg_basket_value  total_quantity
household_key                                                                
1                    5         86   4330.16         50.350698            1997
2                   43         45   1954.34         43.429778             834
3                    8         47   2653.21         56.451277            8540
4                   84         30   1200.11         40.003667             382
5                    8         40    779.06         19.476500             245


In [39]:
customer["avg_quantity"] = customer["total_quantity"] / customer["frequency"]

In [40]:
discount = transaction.groupby("household_key")["RETAIL_DISC"].sum()
customer["total_discount"] = discount

In [41]:
customer["discount_rate"] = customer["total_discount"] / customer["monetary"]

In [42]:
customer.head()

,recency,frequency,monetary,avg_basket_value,total_quantity,avg_quantity,total_discount,discount_rate
household_key,,,,,,,,
1,5,86,4330.16,50.350698,1997,23.220930,-697.04,-0.160973
2,43,45,1954.34,43.429778,834,18.533333,-334.99,-0.171408
3,8,47,2653.21,56.451277,8540,181.702128,-675.16,-0.254469
4,84,30,1200.11,40.003667,382,12.733333,-115.65,-0.096366
5,8,40,779.06,19.476500,245,6.125000,-118.33,-0.151888


In [43]:
customer.shape

(2500, 8)

In [47]:
customer.isnull().sum()

recency             0
frequency           0
monetary            0
avg_basket_value    0
total_quantity      0
avg_quantity        0
total_discount      0
discount_rate       0
dtype: int64

In [50]:
unique_products = transaction.groupby("household_key")["PRODUCT_ID"].nunique()
customer["unique_products"] = unique_products
customer["unique_products"].head()

household_key
1    677
2    546
3    516
4    164
5    199
Name: unique_products, dtype: int64

In [51]:
shopping_days = transaction.groupby("household_key")["DAY"].nunique()
customer["shopping_days"] = shopping_days

In [52]:
customer["spend_per_day"] = customer["monetary"] / customer["shopping_days"]

In [53]:
coupon_redempt = pd.read_csv("../data/raw/coupon_redempt.csv")

In [54]:
coupon_redempt.head()

,household_key,DAY,COUPON_UPC,CAMPAIGN
0,1,421,10000085364,8
1,1,421,51700010076,8
2,1,427,54200000033,8
3,1,597,10000085476,18
4,1,597,54200029176,18


In [59]:
coupon_redempt.groupby("household_key")["COUPON_UPC"].size()

household_key
1        5
8        1
13      21
14       3
18       8
        ..
2488    11
2489    28
2494     5
2496    11
2500     3
Name: COUPON_UPC, Length: 434, dtype: int64

In [55]:
coupon_count = coupon_redempt.groupby("household_key").size()
customer["coupon_redemptions"] = coupon_count

In [63]:
customer["coupon_redemptions"] = customer["coupon_redemptions"].fillna(0)

In [64]:
customer["coupon_rate"] = customer["coupon_redemptions"] / customer["frequency"]

In [65]:
customer.to_csv("../data/processed/customer_features_v1.csv")

In [67]:
customer.shape


(2500, 13)

In [68]:
customer.head()

,recency,frequency,monetary,avg_basket_value,total_quantity,avg_quantity,total_discount,discount_rate,unique_products,shopping_days,spend_per_day,coupon_redemptions,coupon_rate
household_key,,,,,,,,,,,,,
1,5,86,4330.16,50.350698,1997,23.220930,-697.04,-0.160973,677,79,54.812152,5.0,0.05814
2,43,45,1954.34,43.429778,834,18.533333,-334.99,-0.171408,546,45,43.429778,0.0,0.00000
3,8,47,2653.21,56.451277,8540,181.702128,-675.16,-0.254469,516,46,57.678478,0.0,0.00000
4,84,30,1200.11,40.003667,382,12.733333,-115.65,-0.096366,164,30,40.003667,0.0,0.00000
5,8,40,779.06,19.476500,245,6.125000,-118.33,-0.151888,199,33,23.607879,0.0,0.00000


In [69]:
customer.isnull().sum()

recency               0
frequency             0
monetary              0
avg_basket_value      0
total_quantity        0
avg_quantity          0
total_discount        0
discount_rate         0
unique_products       0
shopping_days         0
spend_per_day         0
coupon_redemptions    0
coupon_rate           0
dtype: int64

In [70]:
customer["total_discount"] = customer["total_discount"].abs()

In [71]:
customer["discount_rate"] = customer["total_discount"] / customer["monetary"]

In [72]:
customer[["total_discount", "discount_rate"]].head()

,total_discount,discount_rate
household_key,,
1,697.04,0.160973
2,334.99,0.171408
3,675.16,0.254469
4,115.65,0.096366
5,118.33,0.151888


In [73]:
demographics = pd.read_csv("../data/raw/hh_demographic.csv")

In [74]:
demographics.head()

,classification_1,classification_2,classification_3,HOMEOWNER_DESC,classification_5,classification_4,KID_CATEGORY_DESC,household_key
0,Age Group6,X,Level4,Homeowner,Group5,2,None/Unknown,1
1,Age Group4,X,Level5,Homeowner,Group5,2,None/Unknown,7
2,Age Group2,Y,Level3,Unknown,Group4,3,1,8
3,Age Group2,Y,Level6,Homeowner,Group4,4,2,13
4,Age Group4,Z,Level5,Homeowner,Group3,1,None/Unknown,16


In [75]:

demographics.columns

Index(['classification_1', 'classification_2', 'classification_3',
       'HOMEOWNER_DESC', 'classification_5', 'classification_4',
       'KID_CATEGORY_DESC', 'household_key'],
      dtype='str')

In [76]:
demographics.shape

(801, 8)

In [77]:
demographics.isnull().sum()

classification_1     0
classification_2     0
classification_3     0
HOMEOWNER_DESC       0
classification_5     0
classification_4     0
KID_CATEGORY_DESC    0
household_key        0
dtype: int64

In [ ]:

demographics.info()

<class 'pandas.DataFrame'>
RangeIndex: 801 entries, 0 to 800
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   classification_1   801 non-null    str  
 1   classification_2   801 non-null    str  
 2   classification_3   801 non-null    str  
 3   HOMEOWNER_DESC     801 non-null    str  
 4   classification_5   801 non-null    str  
 5   classification_4   801 non-null    str  
 6   KID_CATEGORY_DESC  801 non-null    str  
 7   household_key      801 non-null    int64
dtypes: int64(1), str(7)
memory usage: 82.5 KB


In [81]:
demographics.head()

,classification_1,classification_2,classification_3,HOMEOWNER_DESC,classification_5,classification_4,KID_CATEGORY_DESC,household_key
0,Age Group6,X,Level4,Homeowner,Group5,2,None/Unknown,1
1,Age Group4,X,Level5,Homeowner,Group5,2,None/Unknown,7
2,Age Group2,Y,Level3,Unknown,Group4,3,1,8
3,Age Group2,Y,Level6,Homeowner,Group4,4,2,13
4,Age Group4,Z,Level5,Homeowner,Group3,1,None/Unknown,16


In [84]:
print(demographics["classification_1"].unique())

<ArrowStringArray>
['Age Group6', 'Age Group4', 'Age Group2', 'Age Group3', 'Age Group1',
 'Age Group5']
Length: 6, dtype: str


In [85]:
print(demographics["classification_2"].unique())

<ArrowStringArray>
['X', 'Y', 'Z']
Length: 3, dtype: str


In [86]:
print(demographics["classification_3"].unique())

<ArrowStringArray>
[ 'Level4',  'Level5',  'Level3',  'Level6',  'Level1',  'Level7',  'Level2',
  'Level8',  'Level9', 'Level12', 'Level10', 'Level11']
Length: 12, dtype: str


In [87]:
print(demographics["HOMEOWNER_DESC"].unique())

<ArrowStringArray>
['Homeowner', 'Unknown', 'Renter', 'Probable Renter', 'Probable Owner']
Length: 5, dtype: str


In [88]:
print(demographics["classification_5"].unique())

<ArrowStringArray>
['Group5', 'Group4', 'Group3', 'Group6', 'Group2', 'Group1']
Length: 6, dtype: str


In [89]:
print(demographics["classification_4"].unique())

<ArrowStringArray>
['2', '3', '4', '1', '5+']
Length: 5, dtype: str


In [90]:
print(demographics["KID_CATEGORY_DESC"].unique())

<ArrowStringArray>
['None/Unknown', '1', '2', '3+']
Length: 4, dtype: str


In [91]:
customer = customer.reset_index()

In [92]:
customer = customer.merge(demographics, on="household_key", how="left")

In [93]:
customer.shape

(2500, 21)

In [94]:
customer.isnull().sum()

household_key            0
recency                  0
frequency                0
monetary                 0
avg_basket_value         0
total_quantity           0
avg_quantity             0
total_discount           0
discount_rate            0
unique_products          0
shopping_days            0
spend_per_day            0
coupon_redemptions       0
coupon_rate              0
classification_1      1699
classification_2      1699
classification_3      1699
HOMEOWNER_DESC        1699
classification_5      1699
classification_4      1699
KID_CATEGORY_DESC     1699
dtype: int64

In [96]:
demographic_columns = [
    "classification_1",
    "classification_2",
    "classification_3",
    "HOMEOWNER_DESC",
    "classification_5",
    "classification_4",
    "KID_CATEGORY_DESC"
]

for column in demographic_columns:
    customer[column] = customer[column].fillna("Unknown")

In [97]:
customer.isnull().sum()

household_key         0
recency               0
frequency             0
monetary              0
avg_basket_value      0
total_quantity        0
avg_quantity          0
total_discount        0
discount_rate         0
unique_products       0
shopping_days         0
spend_per_day         0
coupon_redemptions    0
coupon_rate           0
classification_1      0
classification_2      0
classification_3      0
HOMEOWNER_DESC        0
classification_5      0
classification_4      0
KID_CATEGORY_DESC     0
dtype: int64

In [98]:
customer.to_csv("../data/processed/customer_features_v2.csv", index=False)

In [100]:
campaign = pd.read_csv("../data/raw/campaign_table.csv")
campaign_desc = pd.read_csv("../data/raw/campaign_desc.csv")
coupon = pd.read_csv("../data/raw/coupon.csv")

In [101]:
print(campaign.columns)

Index(['DESCRIPTION', 'household_key', 'CAMPAIGN'], dtype='str')


In [102]:
print(campaign_desc.columns)

Index(['DESCRIPTION', 'CAMPAIGN', 'START_DAY', 'END_DAY'], dtype='str')


In [104]:
print(coupon.columns)

Index(['COUPON_UPC', 'PRODUCT_ID', 'CAMPAIGN'], dtype='str')


In [107]:
campaign.head()
campaign_desc.head()
coupon.head()

,COUPON_UPC,PRODUCT_ID,CAMPAIGN
0,10000089061,27160,4
1,10000089064,27754,9
2,10000089073,28897,12
3,51800009050,28919,28
4,52100000076,28929,25


In [109]:
campaign_count = campaign.groupby("household_key")["CAMPAIGN"].nunique()
customer["campaigns_received"] = campaign_count

In [110]:
customer["campaigns_received"] = customer["campaigns_received"].fillna(0)

In [111]:
customer[["household_key", "campaigns_received"]].head()

,household_key,campaigns_received
0,1,0.0
1,2,8.0
2,3,1.0
3,4,3.0
4,5,1.0


In [ ]:
print(campaign_desc[["CAMPAIGN", "DESCRIPTION"]].head())

   CAMPAIGN DESCRIPTION
0        24       TypeB
1        15       TypeC
2        25       TypeB
3        20       TypeC
4        23       TypeB


In [117]:
campaign_types = campaign.groupby("household_key")["DESCRIPTION"].nunique()
customer["campaign_types"] = campaign_types
customer["campaign_types"] = customer["campaign_types"].fillna(0)

In [118]:
customer["responded_to_campaign"] = 0

In [119]:
customer.loc[customer["coupon_redemptions"] > 0, "responded_to_campaign"] = 1

In [120]:
customer["responded_to_campaign"].value_counts()

responded_to_campaign
0    2066
1     434
Name: count, dtype: int64

In [121]:
customer.to_csv("../data/processed/customer_features_v3.csv", index=False)

In [122]:
customer.shape

(2500, 24)

In [123]:
campaign_count = campaign.groupby("household_key")["CAMPAIGN"].nunique()
customer["campaigns_received"] = campaign_count
customer["campaigns_received"] = customer["campaigns_received"].fillna(0)

In [125]:
print(campaign_desc[["CAMPAIGN", "DESCRIPTION"]].head())

   CAMPAIGN DESCRIPTION
0        24       TypeB
1        15       TypeC
2        25       TypeB
3        20       TypeC
4        23       TypeB


In [126]:
campaign_types = campaign.groupby("household_key")["DESCRIPTION"].nunique()
customer["campaign_types"] = campaign_types
customer["campaign_types"] = customer["campaign_types"].fillna(0)

In [127]:
customer["responded_to_campaign"] = 0
customer.loc[customer["coupon_redemptions"] > 0, "responded_to_campaign"] = 1

In [130]:
customer.shape
customer.head()

,household_key,recency,frequency,monetary,avg_basket_value,total_quantity,avg_quantity,total_discount,discount_rate,unique_products,...,classification_1,classification_2,classification_3,HOMEOWNER_DESC,classification_5,classification_4,KID_CATEGORY_DESC,campaigns_received,campaign_types,responded_to_campaign
0,1,5,86,4330.16,50.350698,1997,23.220930,697.04,0.160973,677,...,Age Group6,X,Level4,Homeowner,Group5,2,None/Unknown,0.0,0.0,1
1,2,43,45,1954.34,43.429778,834,18.533333,334.99,0.171408,546,...,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,8.0,3.0,0
2,3,8,47,2653.21,56.451277,8540,181.702128,675.16,0.254469,516,...,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,1.0,1.0,0
3,4,84,30,1200.11,40.003667,382,12.733333,115.65,0.096366,164,...,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,3.0,2.0,0
4,5,8,40,779.06,19.476500,245,6.125000,118.33,0.151888,199,...,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,1.0,1.0,0


In [131]:
customer["coupon_redeemer"] = customer["responded_to_campaign"]

In [132]:
customer = customer.drop("responded_to_campaign", axis=1)

In [133]:
customer["campaign_rate"] = customer["campaigns_received"] / customer["frequency"]

In [135]:
customer["coupon_per_campaign"] = customer["coupon_redemptions"] / customer["campaigns_received"]

In [136]:
customer["coupon_per_campaign"] = customer["coupon_per_campaign"].fillna(0)

In [137]:
customer[[
    "campaigns_received",
    "campaign_types",
    "coupon_redemptions",
    "campaign_rate",
    "coupon_per_campaign"
]].head()

,campaigns_received,campaign_types,coupon_redemptions,campaign_rate,coupon_per_campaign
0,0.0,0.0,5.0,0.000000,inf
1,8.0,3.0,0.0,0.177778,0.0
2,1.0,1.0,0.0,0.021277,0.0
3,3.0,2.0,0.0,0.100000,0.0
4,1.0,1.0,0.0,0.025000,0.0


In [138]:
customer.shape

(2500, 26)

In [141]:
customer.to_csv("../data/processed/customer_features_v4.csv", index=False)

In [142]:
customer["coupon_per_campaign"] = customer["coupon_per_campaign"].replace(float("inf"), 0)

In [143]:
customer["coupon_per_campaign"] = customer["coupon_per_campaign"].replace(float("-inf"), 0)

In [144]:
customer[[
    "campaigns_received",
    "coupon_redemptions",
    "coupon_per_campaign"
]].head()

,campaigns_received,coupon_redemptions,coupon_per_campaign
0,0.0,5.0,0.0
1,8.0,0.0,0.0
2,1.0,0.0,0.0
3,3.0,0.0,0.0
4,1.0,0.0,0.0


In [145]:
customer.to_csv("../data/processed/customer_features_v4.csv", index=False)

In [146]:
customer.isnull().sum().sum()

np.int64(0)

In [147]:
customer.shape

(2500, 26)

In [149]:
product = pd.read_csv("../data/raw/product.csv")

In [151]:
product.head()
product.columns


Index(['PRODUCT_ID', 'MANUFACTURER', 'DEPARTMENT', 'BRAND', 'COMMODITY_DESC',
       'SUB_COMMODITY_DESC', 'CURR_SIZE_OF_PRODUCT'],
      dtype='str')

In [152]:
product.head()

,PRODUCT_ID,MANUFACTURER,DEPARTMENT,BRAND,COMMODITY_DESC,SUB_COMMODITY_DESC,CURR_SIZE_OF_PRODUCT
0,25671,2,GROCERY,National,FRZN ICE,ICE - CRUSHED/CUBED,22 LB
1,26081,2,MISC. TRANS.,National,NO COMMODITY DESCRIPTION,NO SUBCOMMODITY DESCRIPTION,
2,26093,69,PASTRY,Private,BREAD,BREAD:ITALIAN/FRENCH,
3,26190,69,GROCERY,Private,FRUIT - SHELF STABLE,APPLE SAUCE,50 OZ
4,26355,69,GROCERY,Private,COOKIES/CONES,SPECIALTY COOKIES,14 OZ


In [153]:
transaction_product = transaction.merge(
    product[["PRODUCT_ID", "DEPARTMENT", "COMMODITY_DESC", "BRAND"]],
    on="PRODUCT_ID",
    how="left"
)

In [154]:
transaction_product.head()

,household_key,BASKET_ID,DAY,PRODUCT_ID,QUANTITY,SALES_VALUE,STORE_ID,RETAIL_DISC,TRANS_TIME,WEEK_NO,COUPON_DISC,COUPON_MATCH_DISC,DEPARTMENT,COMMODITY_DESC,BRAND
0,2375,26984851472,1,1004906,1,1.39,364,-0.60,1631,1,0.0,0.0,PRODUCE,POTATOES,Private
1,2375,26984851472,1,1033142,1,0.82,364,0.00,1631,1,0.0,0.0,PRODUCE,ONIONS,National
2,2375,26984851472,1,1036325,1,0.99,364,-0.30,1631,1,0.0,0.0,PRODUCE,VEGETABLES - ALL OTHERS,Private
3,2375,26984851472,1,1082185,1,1.21,364,0.00,1631,1,0.0,0.0,PRODUCE,TROPICAL FRUIT,National
4,2375,26984851472,1,8160430,1,1.50,364,-0.39,1631,1,0.0,0.0,PRODUCE,ORGANICS FRUIT & VEGETABLES,Private


In [155]:
departments = transaction_product.groupby("household_key")["DEPARTMENT"].nunique()
customer["unique_departments"] = departments

In [156]:
categories = transaction_product.groupby("household_key")["COMMODITY_DESC"].nunique()
customer["unique_categories"] = categories

In [ ]:
brands = transaction_product.groupby("household_key")["BRAND"].nunique()
customer["unique_brands"] = brands

In [158]:
customer["category_diversity"] = customer["unique_categories"] / customer["frequency"]

In [159]:
customer[[
    "unique_products",
    "unique_departments",
    "unique_categories",
    "unique_brands",
    "category_diversity"
]].head()

,unique_products,unique_departments,unique_categories,unique_brands,category_diversity
0,677,NaN,NaN,NaN,NaN
1,546,13.0,128.0,2.0,2.844444
2,516,12.0,141.0,2.0,3.000000
3,164,12.0,110.0,2.0,3.666667
4,199,9.0,63.0,2.0,1.575000


In [160]:
customer.shape

(2500, 30)

In [162]:
customer.isnull().sum()

household_key          0
recency                0
frequency              0
monetary               0
avg_basket_value       0
total_quantity         0
avg_quantity           0
total_discount         0
discount_rate          0
unique_products        0
shopping_days          0
spend_per_day          0
coupon_redemptions     0
coupon_rate            0
classification_1       0
classification_2       0
classification_3       0
HOMEOWNER_DESC         0
classification_5       0
classification_4       0
KID_CATEGORY_DESC      0
campaigns_received     0
campaign_types         0
coupon_redeemer        0
campaign_rate          0
coupon_per_campaign    0
unique_departments     1
unique_categories      1
unique_brands          1
category_diversity     1
dtype: int64

In [163]:
customer["unique_departments"] = customer["unique_departments"].fillna(0)
customer["unique_categories"] = customer["unique_categories"].fillna(0)
customer["unique_brands"] = customer["unique_brands"].fillna(0)
customer["category_diversity"] = customer["category_diversity"].fillna(0)

In [164]:
customer.isnull().sum().sum()


np.int64(0)

In [165]:
customer.to_csv("../data/processed/customer_features_final.csv", index=False)

In [166]:
customer.shape

(2500, 30)